# The Perceptron and k-Means (CSC 422)

**Duration:** 90 minutes
**Format:** Live coding with student participation
**Course:** CSC 422 - Machine and Deep Learning

---

## Learning Goals

By the end of class, students should:
- Run the **perceptron** update rule by hand and know why a whole clean pass means it has converged
- Explain what the weight vector and bias mean geometrically (they are a line)
- State the one thing a perceptron **cannot** learn, and why that mattered historically
- Run **Lloyd's algorithm** for k-means by hand for one round
- Explain why k-means only finds a *local* optimum, and what practitioners do about it

---

## Timeline

- **0-40 min** - The Perceptron
- **40-80 min** - k-Means
- **80-90 min** - Wrap-up and connection to PS1

---

**Instructor note:** both algorithms are on PS1 (due Fri of week 3). The numbers
here are deliberately *different* from the pset numbers so students still have
to do the work, but the shape of every calculation is identical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Ready.")

---
# 0-40 min: The Perceptron

**Instructor Notes:**
- This is the oldest learning algorithm in the course (Rosenblatt, 1958) and the
  direct ancestor of every neural network we build in Module 03
- Emphasise: it does not solve any equations. It guesses, and when it is wrong
  it nudges itself toward the point it got wrong. That is *all* it does
- Ask before showing code: "if the model is wrong on a point, which direction
  should the weights move?"

## The model

A perceptron scores an input $\mathbf{x}$ with

$$\text{score} = \mathbf{w} \cdot \mathbf{x} + b$$

and predicts $+1$ if the score is positive, $-1$ otherwise. So $\mathbf{w}$ and
$b$ describe a **line** (in 2D): everything on one side is $+1$, everything on
the other is $-1$.

## The learning rule

Walk through the training points in order. For each one:

> if $y(\mathbf{w} \cdot \mathbf{x} + b) \le 0$, then
> $\mathbf{w} \leftarrow \mathbf{w} + y\,\mathbf{x}$ and $b \leftarrow b + y$

That condition says "the true label and the score disagree in sign" - i.e. we
got it wrong. A score of exactly $0$ counts as wrong: the point is sitting
right on the line.

**Why does adding $y\mathbf{x}$ help?** If the true label is $+1$ and the score
was too low, adding $\mathbf{x}$ to $\mathbf{w}$ increases
$\mathbf{w}\cdot\mathbf{x}$ next time. It literally leans the line toward
that point.

## Do it by hand first

**Instructor Notes:**
- Do this on the board *before* running the code. It takes about six minutes
- Have the class call out the score for each point
- The key moment is the very first point: the score is 0, which counts as wrong

Four points, two features each:

| point | $\mathbf{x}$ | $y$ |
|---|---|---|
| 1 | $(2, 0)$ | $+1$ |
| 2 | $(1, 3)$ | $+1$ |
| 3 | $(-1, 1)$ | $-1$ |
| 4 | $(0, -2)$ | $-1$ |

Start from $\mathbf{w} = (0,0)$, $b = 0$.

In [ ]:
X = np.array([[2, 0], [1, 3], [-1, 1], [0, -2]], dtype=float)
y = np.array([1, 1, -1, -1], dtype=float)

def perceptron_train(X, y, max_passes=10, verbose=True):
    """Train a perceptron. Returns w, b, and the pass it converged on."""
    w = np.zeros(X.shape[1])
    b = 0.0
    for p in range(1, max_passes + 1):
        updates = 0
        if verbose:
            print(f"\n--- Pass {p} ---")
        for xi, yi in zip(X, y):
            score = w @ xi + b
            if yi * score <= 0:                 # wrong (0 counts as wrong)
                w = w + yi * xi                 # lean toward this point
                b = b + yi
                updates += 1
                if verbose:
                    print(f"  x=({xi[0]:g}, {xi[1]:g}) y={yi:+.0f}  score={score:+.0f}  "
                          f"WRONG -> w=({w[0]:g}, {w[1]:g}), b={b:g}")
            elif verbose:
                print(f"  x=({xi[0]:g}, {xi[1]:g}) y={yi:+.0f}  score={score:+.0f}  ok")
        if updates == 0:
            if verbose:
                print(f"  no updates on pass {p} -> CONVERGED")
            return w, b, p
    return w, b, None

w, b, converged_on = perceptron_train(X, y)
print(f"\nFinal: w = ({w[0]:g}, {w[1]:g}), b = {b:g}  (converged on pass {converged_on})")

**Discussion Break (2 minutes)**

- Point 2 was never wrong. Did it contribute anything to the final answer?
- We stopped because a whole pass produced no updates. Why is that a safe
  stopping rule - what would happen if we did one more pass?
- The final $\mathbf{w}$ and $b$ give the line $3x_1 + x_2 - 1 = 0$. Which side
  is $+1$?

## Watch the line move

**Instructor Notes:**
- This is the payoff slide. Each update rotates/shifts the line
- Point out that the line only moves when a point is misclassified

In [ ]:
def boundary_after_each_update(X, y, max_passes=10):
    """Record (w, b) after every update so we can draw them."""
    w, b = np.zeros(X.shape[1]), 0.0
    snapshots = [(w.copy(), b)]
    for _ in range(max_passes):
        updates = 0
        for xi, yi in zip(X, y):
            if yi * (w @ xi + b) <= 0:
                w, b = w + yi * xi, b + yi
                updates += 1
                snapshots.append((w.copy(), b))
        if updates == 0:
            break
    return snapshots

snaps = boundary_after_each_update(X, y)

fig, ax = plt.subplots(figsize=(6.5, 6))
xs = np.linspace(-3, 4, 100)
for i, (wi, bi) in enumerate(snaps):
    if abs(wi[1]) < 1e-9:
        continue                     # vertical line, skip for clarity
    ax.plot(xs, -(wi[0] * xs + bi) / wi[1],
            alpha=0.35 if i < len(snaps) - 1 else 1.0,
            lw=1.2 if i < len(snaps) - 1 else 2.5,
            color="gray" if i < len(snaps) - 1 else "crimson",
            label="final boundary" if i == len(snaps) - 1 else None)

ax.scatter(X[y > 0][:, 0], X[y > 0][:, 1], s=120, marker="o",
           edgecolor="black", facecolor="white", zorder=3, label="+1")
ax.scatter(X[y < 0][:, 0], X[y < 0][:, 1], s=120, marker="s",
           edgecolor="crimson", facecolor="white", zorder=3, label="-1")
ax.set_xlim(-3, 4); ax.set_ylim(-3, 4)
ax.axhline(0, color="lightgray", lw=0.8); ax.axvline(0, color="lightgray", lw=0.8)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title("Every intermediate boundary (grey) and the final one (red)")
ax.legend(loc="lower right"); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## The same thing in scikit-learn

**Instructor Notes:**
- Same API as everything in IC_3: `.fit()`, `.predict()`
- sklearn shuffles and uses a learning rate, so the exact weights differ - the
  *boundary* is what matters

In [ ]:
from sklearn.linear_model import Perceptron

clf = Perceptron(max_iter=1000, tol=None, random_state=0)
clf.fit(X, y)
print("sklearn w:", clf.coef_[0], " b:", clf.intercept_[0])
print("our    w:", w, " b:", b)
print("\nsklearn predictions:", clf.predict(X))
print("true labels:        ", y.astype(int))

## What a perceptron cannot do

**Instructor Notes:**
- This is the historically important part. Minsky and Papert, 1969
- Run the cell and let it fail to converge. Then draw XOR on the board and ask
  the class to find a separating line. They cannot
- **Land the point:** this is a limitation of the *model*, not the training
  algorithm. More data will not help. More passes will not help. The fix is to
  stack two of these with a nonlinearity between them - which is Module 03

XOR: $(0,0) \to -1$, $(1,1) \to -1$, $(0,1) \to +1$, $(1,0) \to +1$.

In [ ]:
X_xor = np.array([[0, 0], [1, 1], [0, 1], [1, 0]], dtype=float)
y_xor = np.array([-1, -1, 1, 1], dtype=float)

w_x, b_x, conv = perceptron_train(X_xor, y_xor, max_passes=8, verbose=False)
print("Converged on pass:", conv)          # None -> never converged
print("Predictions:", np.sign(X_xor @ w_x + b_x).astype(int))
print("True labels:", y_xor.astype(int))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(X_xor[y_xor > 0][:, 0], X_xor[y_xor > 0][:, 1], s=180, marker="o",
           edgecolor="black", facecolor="white", zorder=3, label="+1")
ax.scatter(X_xor[y_xor < 0][:, 0], X_xor[y_xor < 0][:, 1], s=180, marker="s",
           edgecolor="crimson", facecolor="white", zorder=3, label="-1")
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_title("XOR - try to separate these with one straight line")
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

**Discussion Break (3 minutes)**

- Try to draw a single straight line with the two circles on one side and the
  two squares on the other. Convince yourself it is impossible
- Here is the proof in one line. The two $+1$ points need $w_2 + b > 0$ and
  $w_1 + b > 0$; adding those gives $w_1 + w_2 + 2b > 0$. The two $-1$ points
  need $b < 0$ and $w_1 + w_2 + b < 0$; adding those gives
  $w_1 + w_2 + 2b < 0$. The same quantity cannot be both
- **What would fix it?** (Answer: a hidden layer. That is Module 03.)

---
# 40-80 min: k-Means

**Instructor Notes:**
- First **unsupervised** algorithm of the course - there are no labels anywhere
- Contrast with everything so far: no $y$, so no loss comparing prediction to
  truth. Instead we minimise how spread out each cluster is
- Ask: "if I gave you six dots on the board and said 'make two groups', what
  would you do?" Their answer usually *is* Lloyd's algorithm

## Lloyd's algorithm

Pick $k$. Put $k$ centres down somewhere. Then repeat two steps until nothing
changes:

1. **Assign** - each point joins the nearest centre
2. **Update** - each centre moves to the average of the points that joined it

That is the whole algorithm.

## One round by hand

**Instructor Notes:**
- Do round 1 on the board before running anything
- Deliberately start both centres in the same corner so students see the centres
  travel. Ask them to predict where $\mathbf{c}_2$ will end up

Six points: $(1,2), (2,1), (2,3), (7,6), (8,7), (8,5)$, with $k = 2$ and a
deliberately poor start: $\mathbf{c}_1 = (1,2)$, $\mathbf{c}_2 = (2,1)$.

In [ ]:
P = np.array([[1, 2], [2, 1], [2, 3], [7, 6], [8, 7], [8, 5]], dtype=float)

def kmeans(P, centres, max_rounds=10, verbose=True):
    """Lloyd's algorithm. Returns final centres, assignments, history."""
    C = np.array(centres, dtype=float)
    history = [C.copy()]
    for r in range(1, max_rounds + 1):
        # 1. assign: squared distance from every point to every centre
        d2 = ((P[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        assign = d2.argmin(axis=1)
        # 2. update: each centre becomes the mean of its own points
        C_new = np.array([P[assign == j].mean(axis=0) if (assign == j).any() else C[j]
                          for j in range(len(C))])
        if verbose:
            print(f"round {r}: assignments = {assign}")
            for j, c in enumerate(C_new):
                print(f"          c{j+1} -> ({c[0]:.3g}, {c[1]:.3g})")
        if np.allclose(C, C_new):
            if verbose:
                print(f"          centres did not move -> CONVERGED")
            return C_new, assign, history
        C = C_new
        history.append(C.copy())
    return C, assign, history

C_final, assign, history = kmeans(P, [[1, 2], [2, 1]])

**Discussion Break (2 minutes)**

- In round 1, $\mathbf{c}_2$ jumped to roughly $(6.25, 4.75)$ - a spot where
  **no data point sits at all**. Why?
- Which point changed clusters between round 1 and round 2?
- How do we know we are finished?

In [ ]:
fig, axes = plt.subplots(1, len(history), figsize=(4.2 * len(history), 4))
if len(history) == 1:
    axes = [axes]
colors = ["#14655C", "#E2513A"]

for r, (ax, C_r) in enumerate(zip(axes, history)):
    d2 = ((P[:, None, :] - C_r[None, :, :]) ** 2).sum(axis=2)
    a = d2.argmin(axis=1)
    for j in range(len(C_r)):
        ax.scatter(P[a == j][:, 0], P[a == j][:, 1], s=90, color=colors[j], zorder=3)
        ax.scatter(*C_r[j], marker="D", s=170, color=colors[j],
                   edgecolor="black", linewidth=1.2, zorder=4)
    ax.set_title(f"centres at start of round {r+1}")
    ax.set_xlim(0, 9); ax.set_ylim(0, 9); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("Final centres:")
for j, c in enumerate(C_final):
    print(f"  c{j+1} = ({c[0]:.4g}, {c[1]:.4g})")

## The same thing in scikit-learn

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(P)
print("sklearn centres:\n", np.round(km.cluster_centers_, 4))
print("\nours:\n", np.round(C_final, 4))
print("\nlabels:", km.labels_)

## k-means only finds a *local* optimum

**Instructor Notes:**
- This is the single most important caveat and it is easy to demonstrate
- Run it and let the number land. 101 versus 1.5 is a shocking gap
- Then explain `n_init` in the sklearn call above - that is exactly what it is
  defending against

Six points on a line: $0, 1, 10, 11, 20, 21$, with $k = 3$.

Any human would group them $\{0,1\}, \{10,11\}, \{20,21\}$. Watch what
happens if the centres start at $0$, $1$ and $15.5$.

In [ ]:
L = np.array([[0], [1], [10], [11], [20], [21]], dtype=float)

def wcss(P, C, assign):
    """Within-cluster sum of squares - the thing k-means is minimising."""
    return sum(((P[assign == j] - C[j]) ** 2).sum() for j in range(len(C)))

for name, start in [("bad start  (0, 1, 15.5)", [[0], [1], [15.5]]),
                    ("good start (0, 10, 20) ", [[0], [10], [20]])]:
    C_f, a_f, _ = kmeans(L, start, verbose=False)
    print(f"{name}: centres = {np.round(C_f.ravel(), 3)}  "
          f"clusters = {a_f}  WCSS = {wcss(L, C_f, a_f):.4g}")

print("\nBoth are stable - neither will move again. One is 67x worse than the other,")
print("and k-means has no way to tell.")

**Discussion Break (3 minutes)**

- Verify the bad start really is stuck: point $0$ is alone with centre $0$, point
  $1$ is alone with centre $1$, and $\{10,11,20,21\}$ averages to exactly
  $15.5$. Nothing moves
- So what do we do about it? (Run it many times from different random starts and
  keep the best - that is sklearn's `n_init`. Or start smartly, which is
  `k-means++`)
- We minimised WCSS. What happens to WCSS if we let $k$ grow to 6? Is
  "minimise WCSS" a good way to *choose* $k$?

---
# 80-90 min: Wrap-up

## What you should walk out with

| | perceptron | k-means |
|---|---|---|
| supervised? | yes - needs labels $y$ | no - no labels anywhere |
| what it learns | a line $\mathbf{w}\cdot\mathbf{x} + b = 0$ | $k$ centres |
| how it learns | nudge toward misclassified points | assign, then average; repeat |
| stops when | a whole pass makes no updates | assignments stop changing |
| main gotcha | only works if the data is linearly separable (XOR) | only a local optimum; depends on the start |

## On PS1

Both algorithms show up on **PS1, due Friday**. The numbers there are different
from today's, but every calculation has the same shape as one we just did:

- **Problem 4** runs the perceptron update rule by hand on four points, exactly
  like the by-hand trace at the start of today
- **Problem 7** runs one round of Lloyd's algorithm by hand, exactly like the
  round we did on the board

If you can redo today's two by-hand traces without looking, you can do both
problems.

## Looking ahead

- The perceptron's XOR failure is the reason **Module 03 (Neural Networks)**
  exists. A hidden layer is precisely the fix
- k-means is your first taste of **Module 02 (Structure Without Labels)**, where
  we come back to it properly alongside PCA